# Functions for analysis notebook

packages used: 
- pandas (version 3.0.3)
- torch (version 2.7.1)
- transformers (version 4.52.4)
- spacy (version 3.8.14)
- geopy (version 2.5.0)
- time / json

## 1- Imports and versions of the packages

In [ ]:
import pandas as pd

from transformers import pipeline
import torch

import spacy

from geopy.geocoders import Nominatim
import time

import json

In [13]:
print("Versions:")
print("pandas: ", pd.__version__)
print("torch: ", torch.__version__)
import transformers
print("transformers: ", transformers.__version__)
print("spacy: ", spacy.__version__)
import geopy
print("geopy: ", geopy.__version__)

Versions:
pandas:  3.0.3
torch:  2.7.1+cpu
transformers:  4.52.4
spacy:  3.8.14
geopy:  2.5.0


## 2- Initialisations

### 2.1- Zero shot classification

- Initialising the classifier object for zero-shot classification (theme classification)

- Setting the default themes



In [ ]:
classifier = pipeline("zero-shot-classification",
                      model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0")

Device set to use cpu


In [ ]:
default_themes = [
    "right to education",
    "right to work",
    "right to life",
    "right to health",
    "access to justice",
    "adequate standard of living"
]

hypothesis_template = "This text describes a violation of the {}."

### 2.2- Country extraction

- loading the spacy module
- initialising the geopy Nominatim module 

In [16]:
nlp = spacy.load('en_core_web_sm')

In [17]:
geolocator = Nominatim(user_agent="africa-country")

## 3- Theme classification functions

In [ ]:
def classification_function(extract, themes=default_themes):
    '''
    input:
    extract: string, text to classify
    themes: list of available themes

    output:
    rel_themes: list of themes in the provided extracts
    '''


    # step 1: returning the classification scores

    print("-"*20)
    print("Starting zero-shot classification")
    print("-"*20)

    results = classifier(extract, themes, hypothesis_template=hypothesis_template, multi_label=True)
    scores = results['scores']  

    # results are returned organised

    # step 2: keeping only the themes that have a classification score above 0.4
    rel_scores = [x for x in scores if x > 0.4]

    n_rel_themes = len(rel_scores)

    print(f"Classification succeeded, {n_rel_themes} identified")

    rel_themes = results['labels'][:n_rel_themes]


    #step 3: returning the relevant themes
    return rel_themes




## 4- Country recognition function

In [19]:
def places_recognition(extract):
    '''
    input:
    extract: string, text containing places names

    output:
    places: list of places mentionned in the extract
    '''

    doc = nlp(extract)

    places = []

    for ent in doc.ents:
        if ent.label_ == "GPE":
            places.append(ent.text)
    
    return places


In [ ]:
def map_place_to_country(place, map_dic):
    '''
    input:
    place: string, place name
    map_dic: dictionnary with places names that have already been mapped

    output:
    country: string, country name of the place
    new_map_dic: dictionnary with places names that have already been mapped
    '''

    new_map_dic = map_dic.copy()

    # option 1: the place has already been mapped
    if place in map_dic.keys():
        print("Place was already mapped")
        country = map_dic[place]
    
    # option 2: the place has not been mapped yet
    else:
        # getting the country name
        print("New place, using geopy")
        location = geolocator.geocode(place, addressdetails=True)

        if location: # only do this if a match has been found
            country = location.raw['address']['country']
        
        else:
            country = None

        # updating the places name dictionnary
        new_map_dic[place] = country

    return country, new_map_dic
    

## 5- analysis functions

In [ ]:
def analyse(data, themes=default_themes, map_dic={}):
    '''
    input:
    data: pandas dataframe, containing the extracts to analyse
    themes: list of default themes to classify the extract in
    map_dic: dictionnary of places that have already been mapped

    output:
    new_data: pandas dataframe, with the themes and country columns filled
    new_map_dic: new mapping dictionnary
    '''

    # step 1: creating a copy of the dataframe and of the mapping dictionnary
    new_data = data.copy()
    new_map_dic = map_dic.copy()

    # resetting the index of the dataframe
    new_data = new_data.reset_index()

    # step 2: adding the theme and places mentionned for each line
    n_lines = len(new_data) # number of lines to analyse

    for i in range(n_lines):
        
        # only analyse if there is nothing in the country and theme column
        if pd.isna(new_data["country"].loc[i]) and pd.isna(new_data["theme"].loc[i]):
            print("-"*20)
            print(f"analysing line {i}")
            print("-"*20)

            extract = new_data["extract"].loc[i]
            title = new_data["title"].loc[i]

            # step 2.a: adding the theme(s) of the extract
            id_themes = classification_function(extract, themes=themes)
            
            new_data.at[i, "theme"] = id_themes

            #step 2.b: adding the countries of the extract
            print("Extracting places names")
            places = places_recognition(extract)

            if len(places) == 0: # if no place has been identified in the extract, test with the title
                places = places_recognition(title)

            print(f"{len(places)} places identified, extracting country names")
            # initialising the country list 
            countries = []

            for place in places:
                country, new_map_dic = map_place_to_country(place, new_map_dic)
                if country not in countries:
                    countries.append(country)
                time.sleep(15) # making sure the quotas are not reached
            
            new_data.at[i, "country"] = countries
            print(f"found {len(countries)} countries")
    
    return new_data, new_map_dic